# Examen Final
## Probabilidad y Estadística

Legajo a2412

Quiroga, Martin Gabriel

Siguiendo con la historia de Don Francisco, con el tiempo y gracias a los análisis de Matías, el pequeño comerciante de barrio cuenta hoy con 5 supermercados: ’Santa Ana’, ’La Floresta’, ’Los Cedros’, ’Palermo’ y ’Córdoba’.
También Matías ha avanzado en la Especialización en Inteligencia Artificial. Un día Don Francisco le plantea algunas inquietudes adicionales:
1. Don Francisco quiere entender mejor las ventas por mes del supermercado ’Santa Ana’.
2. Más aún, Don Francisco no sabe si puede estar seguro de que las ventas son las mismas en todos los supermercados o si hay alguno que se comporte mejor que los demás, y si alguna de las tiendas necesita más atención porque sus ventas sean peores que las de las otras.

Con base en lo anterior,

1. (2.5 puntos) Crear una simulación de las ventas diarias de los almacenes de Don Francisco y de Don Miguel, usando distribuciones Poisson, entre los años 2023, 2024 y 2025. En cada fecha, el parámetro $\lambda_t$ debe ser la suma de los siguientes efectos:

**Efecto anual:**
| Año | Efecto |
| --- | --- |
| 2023 | 1000 |
| 2024 | 1500 |
| 2025 | 2000 |

**Efecto mensual:**
| Mes | Efecto |
| --- | --- |
| Enero | 1000 |
| Febrero | 1500 |
| Marzo | 2000 |
| Abril | 2000 |
| Mayo | 2500 |
| Junio | 2500 |
| Julio | 3000 |
| Agosto | 2500 |
| Septiembre | 2500 |
| Octubre | 2000 |
| Noviembre | 1500 |
| Diciembre | 1000 |

**Efecto diario:**
| Día | Efecto |
| --- | --- |
| Domingo | 1000 |
| Lunes | 2000 |
| Martes | 3000 |
| Miércoles | 3500 |
| Jueves | 3000 |
| Viernes | 2000 |
| Sábado | 1000 |

**Efecto por tienda:**
| Tienda | Efecto |
| --- | --- |
| Santa Ana | 5000 |
| La Floresta | 200 |
| Los Cedros | 3000 |
| Palermo | 1000 |
| Córdoba | 3000 |

2. (2.5 puntos) Con base en los datos generados, determinen intervalos de confianza empíricos para el supermercado ’Santa Ana’ en cada mes, para significancias del 95 % y el 99 %.
3. (2.5 puntos) De igual manera, realicen pruebas ANOVA para determinar si las ventas esperadas de todas las tiendas son iguales o no, con significancia del 95.
4. (2.5 puntos) Finalmente, identifiquen la tienda con mayor promedio de ventas y la tienda con menor promedio de ventas y realicen una prueba de hipótesis para determinar si la diferencia entre ellas es distinta de cero o no. Verifiquen si las tiendas identificadas corresponden a las tiendas con mayores y menores efectos.

In [1]:
import numpy as np
import pandas as pd
import datetime
from scipy.stats import poisson

#### 1) Crear una simulación de las ventas diarias de los almacenes de Don Francisco y de Don Miguel, usando distribuciones Poisson, entre los años 2023, 2024 y 2025.

In [2]:
# EL valor de lambda ahora dependerá de dos "variables": el día (en el que se
# incluye día de la semana, mes y año) así como de la tienda correspondiente, lo
# que nos llevará a tener que calcular un lambda por día, por almacén. Esto 
# sería n_días * n_tiendas = (365 + 366 + 365) * 5 = 5480 valores.
# Entre las posibles alternativas, se decide crear un único dataframe con los 
# 5480 registros.

start_date = datetime.datetime(2023, 1, 1)
end_date = datetime.datetime(2025, 12, 31)
fechas = pd.date_range(start_date, end_date)

tiendas = ['Santa Ana', 'La Floresta', 'Los Cedros', 'Palermo', 'Córdoba']

# Se crea un dataframe multi-índice (día y tienda)
df = pd.MultiIndex.from_product([fechas, tiendas], names=['date', 'store']).to_frame(index=False)
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['dayOfWeek'] = df['date'].dt.day_of_week


In [3]:
# Se crean los diccionarios de efectos
year_effect_map = {
    2023: 1000,
    2024: 1500,
    2025: 2000
}
month_effect_map =  {
    1: 1000,
    2: 1500,
    3: 2000,
    4: 2000,
    5: 2500,
    6: 2500,
    7: 3000,
    8: 2500,
    9: 2500,
    10: 2000,
    11: 1500,
    12: 1000
}
# Usando pandas.DateTime el día 0 corresponde al Lunes y el 6 al Domingo
dow_effect_map = {
    6: 1000,
    0: 2000,
    1: 3000,
    2: 3500,
    3: 3000,
    4: 2000,
    5: 1000
}

store_effect_map = {
    'Santa Ana': 5000,
    'La Floresta': 2000,
    'Los Cedros': 3000,
    'Palermo': 1000,
    'Córdoba':3000
}


df['year_effect'] = df['year'].map(year_effect_map)
df['month_effect'] = df['month'].map(month_effect_map)
df['dow_effect'] = df['dayOfWeek'].map(dow_effect_map)
df['store_effect'] = df['store'].map(store_effect_map)

df['lambda'] = df['year_effect'] + df['month_effect'] + df['dow_effect'] + df['store_effect']

In [4]:
# Ahora sí, simulamos todas las ventas por día y tienda, siguiendo Poisson
rng = np.random.default_rng(seed=42)
df['sales'] = poisson.rvs(mu=df['lambda'], random_state=rng)

In [5]:
# Visualizamos los primeros registros del dataframe generado
df.head(20)

,date,store,year,month,dayOfWeek,year_effect,month_effect,dow_effect,store_effect,lambda,sales
0,2023-01-01,Santa Ana,2023,1,6,1000,1000,1000,5000,8000,8076
1,2023-01-01,La Floresta,2023,1,6,1000,1000,1000,2000,5000,5087
2,2023-01-01,Los Cedros,2023,1,6,1000,1000,1000,3000,6000,5878
3,2023-01-01,Palermo,2023,1,6,1000,1000,1000,1000,4000,4051
4,2023-01-01,Córdoba,2023,1,6,1000,1000,1000,3000,6000,5899
5,2023-01-02,Santa Ana,2023,1,0,1000,1000,2000,5000,9000,8965
6,2023-01-02,La Floresta,2023,1,0,1000,1000,2000,2000,6000,6032
7,2023-01-02,Los Cedros,2023,1,0,1000,1000,2000,3000,7000,6987
8,2023-01-02,Palermo,2023,1,0,1000,1000,2000,1000,5000,5011
9,2023-01-02,Córdoba,2023,1,0,1000,1000,2000,3000,7000,7089


#### 2) Con base en los datos generados, determinen intervalos de confianza empíricos para el supermercado ’Santa Ana’ en cada mes, para significancias del 95 % y el 99 %.

Se debe tomar solo la tienda de Santa Ana y agrupar los datos mensualmente. Luego, para cada mes se deben determinar los intervalos de confianza indicados.

Se toman los datos como empíricos tal que no se conocen los estadísticos de la población (es decir, solo usamos los datos observados sin asumir la forma de la distribución subyacente), pero sí podemos obtener los de la muestra. Dentro de un mismo mes, las observaciones no provienen de una única distribución Poisson, sino de una mezcla: cada día tiene un lambda distinto porque varía el efecto del día de la semana, el efecto del mes y el efecto del año. Por lo tanto, no podemos asumir directamente un modelo paramétrico simple para la distribución mensual.

Por el teorema central del límite, como tendremos aproximadamente 90 datos para cada mes (~30 * 3 años), la distribución de la media muestral se arpoxima lo suficiente a una normal, sin importar la distribución original. Sin embargo, como la varianza poblacional es desconocida, debería utilizar formalmente la ditribución t-Student (aunque con ese tamaño de muestras los cuantiles son prácticamente iguales a los de la normal).

Dicho esto, para determinar los intervalos de confianza, se debe obtener el valor crítico $t^*$ de la tabla t-Student bilateral correspondiente.

In [6]:
# Se filtra por tienda Santa Ana
df_sa = df[df['store'] == 'Santa Ana'].copy()

# Se agrupa por mes, y se calculan la media, la desviación estándar y la cantidad
# de observaciones, para cada uno
sa_mes = df_sa.groupby('month')['sales'].agg(['mean', 'std', 'count']).reset_index()

# Además, se agrega el cálculo de s/sqrt(n) (error estándar) para simplificar
# otros cálculos después
sa_mes['se'] = sa_mes['std'] / np.sqrt(sa_mes['count'])
sa_mes


,month,mean,std,count,se
0,1,9748.752688,1042.957309,93,108.149635
1,2,10215.847059,997.283841,85,108.170620
2,3,10673.526882,976.078997,93,101.214677
3,4,10722.566667,1039.228950,90,109.544350
4,5,11252.612903,989.230630,93,102.578438
5,6,11174.344444,1001.938180,90,105.613557
6,7,11741.483871,1047.583810,93,108.629381
7,8,11220.397849,987.043578,93,102.351651
8,9,11179.733333,1027.860621,90,108.346023
9,10,10734.989247,1022.658019,93,106.044697


Los intervalos a determinar son bilaterales, con $\alpha_1 = 0.05$ y $\alpha_2 = 0.01$, y un $n$ variable por mes, dependiendo de la cantidad de días del mismo. El valor de $t^*$ se deberá buscar entonces para $\alpha_1 /2 = 0.025$ y $\alpha_2 /2 = 0.005$, y $n - 1$, propio de cada mes. Luego, los límites del intervalo, para cada mes $m$, se calculan como:
$$\left[\bar{X_m} - t^* \frac{S_m}{\sqrt{n_m}} ; \bar{X_m} + t^* \frac{S_m}{\sqrt{n_m}}\right]$$

Con Python, podemos obtener buscar el valor $t^*$ usando la librería de `scipy`.

In [7]:
from scipy.stats import t

sig1 = 0.95
sig2 = 0.99
alfa1 = 1 - sig1
alfa2 = 1 - sig2

# El número de observaciones n será para cada mes, se aplica directamente desde
# el dataframe. Obtenemos el t* correspondiente para cada mes
sa_mes['t_95'] = t.ppf(1 - alfa1/2, df=sa_mes['count'] - 1)
sa_mes['t_99'] = t.ppf(1 - alfa2/2, df=sa_mes['count'] - 1)

# Luego, se calculan los límites inferior y superior de los intervalos de
# confianza para cada mes.
sa_mes['t_95_lower'] = sa_mes['mean'] - sa_mes['t_95'] * sa_mes['se']
sa_mes['t_95_upper'] = sa_mes['mean'] + sa_mes['t_95'] * sa_mes['se']
sa_mes['t_99_lower'] = sa_mes['mean'] - sa_mes['t_99'] * sa_mes['se']
sa_mes['t_99_upper'] = sa_mes['mean'] + sa_mes['t_99'] * sa_mes['se']

In [8]:
nombres_meses = {
    1: 'Enero', 2: 'Febrero', 3: 'Marzo', 4: 'Abril',
    5: 'Mayo', 6: 'Junio', 7: 'Julio', 8: 'Agosto',
    9: 'Septiembre', 10: 'Octubre', 11: 'Noviembre', 12: 'Diciembre'
}

print("Los intevalos de confianza del 95% para cada mes son:")
for index, row in sa_mes.iterrows():
    mes_nombre = nombres_meses[int(row['month'])]
    lim_inf = round(row['t_95_lower'], 2)
    lim_sup = round(row['t_95_upper'], 2)
    print(f"{mes_nombre}: [{lim_inf}, {lim_sup}]")

print("\nLos intevalos de confianza del 99% para cada mes son:")
for index, row in sa_mes.iterrows():
    mes_nombre = nombres_meses[int(row['month'])]
    lim_inf = round(row['t_99_lower'], 2)
    lim_sup = round(row['t_99_upper'], 2)
    print(f"{mes_nombre}: [{lim_inf}, {lim_sup}]")

sa_mes

Los intevalos de confianza del 95% para cada mes son:
Enero: [9533.96, 9963.55]
Febrero: [10000.74, 10430.96]
Marzo: [10472.51, 10874.55]
Abril: [10504.9, 10940.23]
Mayo: [11048.88, 11456.34]
Junio: [10964.49, 11384.2]
Julio: [11525.74, 11957.23]
Agosto: [11017.12, 11423.68]
Septiembre: [10964.45, 11395.01]
Octubre: [10524.38, 10945.6]
Noviembre: [9988.31, 10400.16]
Diciembre: [9496.61, 9923.52]

Los intevalos de confianza del 99% para cada mes son:
Enero: [9464.28, 10033.22]
Febrero: [9930.75, 10500.95]
Marzo: [10407.3, 10939.75]
Abril: [10434.22, 11010.91]
Mayo: [10982.8, 11522.43]
Junio: [10896.35, 11452.34]
Julio: [11455.75, 12027.21]
Agosto: [10951.18, 11489.62]
Septiembre: [10894.54, 11464.92]
Octubre: [10456.06, 11013.92]
Noviembre: [9921.44, 10467.03]
Diciembre: [9427.36, 9992.77]


,month,mean,std,count,se,t_95,t_99,t_95_lower,t_95_upper,t_99_lower,t_99_upper
0,1,9748.752688,1042.957309,93,108.149635,1.986086,2.630330,9533.958178,9963.547198,9464.283501,10033.221875
1,2,10215.847059,997.283841,85,108.170620,1.988610,2.635632,10000.737918,10430.956200,9930.749061,10500.945056
2,3,10673.526882,976.078997,93,101.214677,1.986086,2.630330,10472.505797,10874.547967,10407.298920,10939.754843
3,4,10722.566667,1039.228950,90,109.544350,1.986979,2.632204,10504.904377,10940.228956,10434.223570,11010.909763
4,5,11252.612903,989.230630,93,102.578438,1.986086,2.630330,11048.883271,11456.342535,10982.797801,11522.428005
5,6,11174.344444,1001.938180,90,105.613557,1.986979,2.632204,10964.492555,11384.196333,10896.347996,11452.340893
6,7,11741.483871,1047.583810,93,108.629381,1.986086,2.630330,11525.736544,11957.231198,11455.752794,12027.214947
7,8,11220.397849,987.043578,93,102.351651,1.986086,2.630330,11017.118636,11423.677063,10951.179271,11489.616428
8,9,11179.733333,1027.860621,90,108.346023,1.986979,2.632204,10964.452094,11395.014573,10894.544478,11464.922188
9,10,10734.989247,1022.658019,93,106.044697,1.986086,2.630330,10524.375326,10945.603168,10456.056742,11013.921753


Los intervalos para cada mes entonces, son:

| Mes | t_95 | t_99 | t_95_lower | t_95_upper | t_99_lower | t_99_upper |
| --- | --- | --- | --- | --- | --- | --- |
| Enero |	1.9860 | 2.6303 | 9533.96 | 9963.55 | 9464.28 | 10033.22 |
| Febrero | 1.9886 | 2.6356 | 10000.74 | 10430.96 | 9930.75 | 10500.95 |
| Marzo | 1.9860 | 2.6303 | 10472.51 | 10874.55 | 10407.30 | 10939.75 |
| Abril | 1.9869 | 2.6322 | 10504.90 | 10940.23 | 10434.22 | 11010.91 |
| Mayo | 1.9860 | 2.6303 | 11048.88 | 11456.34 | 10982.80 | 11522.43 |
| Junio | 1.9869 | 2.6322 | 10964.49 | 11384.20 | 10896.35 | 11452.34 |
| Julio | 1.9860 | 2.6303 | 11525.74 | 11957.23 | 11455.75 | 12027.21 |
| Agosto | 1.9860 | 2.6303 | 11017.12 | 11423.68 | 10951.18 | 11489.62 |
| Septiembre | 1.9869 | 2.6322 | 10964.45 | 11395.01 | 10894.54 | 11464.92 |
| Octubre | 1.9860 | 2.6303 | 10524.38 | 10945.60 | 10456.06 | 11013.92 |
| Noviembre | 1.9869 | 2.6322 | 9988.31 | 10400.16 | 9921.44 | 10467.03 |
| Diciembre | 1.9860 | 2.6303 | 9496.61 | 9923.52 | 9427.36 | 9992.77 |


#### 3) De igual manera, realicen pruebas ANOVA para determinar si las ventas esperadas de todas las tiendas son iguales o no, con significancia del 95.

Para analizar si las ventas esperadas son significantemente iguales o no, se deben agrupar ahora las ventas por tienda, formando 5 grupos de 1096 observaciones cada uno.

La hipótesis nula será que todas las medias son iguales. Esto es:
$$H_0: \mu_1 = \mu_2 = \mu_3 = \mu_4 = \mu_5$$

Y la hipótesis alternativa, será que existe al menos una media diferente. Esto es:
$$H_1: \exists \mu_i \neq \mu_j \text{ ; para } i, j \in \{1, 2, 3, 4, 5\}$$

Para comprobar las hipótesis, se "descompone" la variabilidad en dos: variabilidad entre grupos y variabilidad dentro de los grupos (o intra grupos). Luego, si la variabilidad entre grupos es mucho mayor que la variabilidad intra grupos, entonces media de ventas de las tiendas se puede considerar realmente diferente.

Definiendo:
- Número de grupos: $k = 5$
- Número de observaciones para la tienda $j$: $n_j$
- Ventas del día $i$ en la tienda $j$: $X_{ij}$
- Media de la tienda $j$: $\bar{X_j}$
- Media global: $\bar{X}$

Se tiene entonces:
$$\sum_{j=1}^{k} \sum_{i=1}^{n_j} (X_{ij} - \bar{X})^2 = \sum_{j=1}^{k} n_j (\bar{X}_j - \bar{X})^2 + \sum_{j=1}^{k} \sum_{i=1}^{n_j} (X_{ij} - \bar{X}_j)^2$$
$$SS_{total} = SS{fact} + SS{error}$$

Los grados de libertad para cada uno son:
- $SS_{total} \rightarrow N - 1$
- $SS_{fact} \rightarrow k - 1$
- $SS_{error} \rightarrow N - k$

Entonces, los cuadrados medios serán:
$$T = \frac{SS_{fact}}{k - 1} \text{ ; } E = \frac{SS_{error}}{N - k}$$

El estadístico de prueba es el coeficiente F, definido por:
$$F = \frac{T}{E}$$

Bajo la hipótesis nula, este estadístico sigue una distribución F de Fisher con grados de libertad correspondientes a $k - 1$ en el numerador y a $N - k$ en denominador, es decir $(k-1, N-k)=(4, 5475)$ grados de libertad.

Para una significacia del 95%, se tiene $\alpha = 0.05$. Ahora se debe encontrar el valor F crítico $F^*$, tal que si $F > F^*$, entonces se rechaza la hipótesis nula. O dicho de otra forma, se rechaza $H_0$ si el p-valor es menor que 0.05.

La tabla de ANOVA es:

| Fuente | Suma de cuadrados | Grados de libertad | Cuadrado medio | F |
| --- | --- | --- | --- | --- |
| Intergrupo | $SS_{fact}$ | $k - 1$ | $T=\frac{SS_{fact}}{k - 1}$ | $F=\frac{T}{E}$ |
| Intragrupo | $SS_{error}$ | $N - k$ | $E=\frac{SS_{error}}{N - k}$ |  |
| Total | $SS_{total}$ | $N - 1$ |  |  |

In [ ]:
from scipy.stats import f
from scipy.stats import f_oneway

In [18]:
# Datos necesarios para los cálculos de ANOVA
X = np.mean(df['sales'])
groups = [group['sales'].values for _, group in df.groupby('store')]
k = len(groups)
N = len(df)

# Calculamos las sumas de cuadrados correspondientes
ss_total = sum((df['sales'] - X)**2)
print(f"Suma de cuadrados total: {ss_total:.2f}")

ss_fact = sum(len(group) * (np.mean(group) - X)**2 for group in groups)
print(f"Suma de cuadrados intergrupos: {ss_fact:.2f}")

ss_error = sum(((group - np.mean(group))**2).sum() for group in groups)
print(f"Suma de cuadrados intragrupos: {ss_error:.2f}")

T = ss_fact / (k - 1)
print(f"T: {T:.2f}")
E = ss_error / (N - k)
print(f"E: {E:.2f}")
F = T / E
print(f"F: {F:.2f}")
p_valor = f.sf(F, k-1, N-k)
print(f"p-valor: {p_valor:.4f}")

Suma de cuadrados total: 17302753891.12
Suma de cuadrados intergrupos: 9643876416.68
Suma de cuadrados intragrupos: 7658877474.44
T: 2410969104.17
E: 1398881.73
F: 1723.50
p-valor: 0.0000


Después de hacer los cálculos, mediante un script, se conforma la tabla correspondiente:

| Fuente | Suma de cuadrados | Grados de libertad | Cuadrado medio | F |
| --- | --- | --- | --- | --- |
| Intergrupo | $SS_{fact}=9643876416.68$ | $k-1=4$ | $T=2410969104.17$ | $F=1723.50$ |
| Intragrupo | $SS_{error}=7658877474.44$ | $N - k=5475$ | $E=1398881.73$ |  |
| Total | $SS_{total}=17302753891.12$ | $N - 1=5479$ |  |  |

Ahora, yendo a la tabla de valores críticos de la distribución F para una significacia del 95%, se observa que no aparece el valor para $(4, 5475)$, pero si se encuentra que $(4, 120) \rightarrow f^*=2.45$ y $(4, \infty) \rightarrow f^*=2.37$, por lo que podemos asegurar que el F crítico se encontrará entre ellos y de cualquier manera tendrá un valor mucho menor que el F obtenido del cálculo ($F = 1723.50$). De esta manera, se rechaza la hipótesis nula $H_0$ de que las ventas esperadas de todas las tiendas sean iguales con una significacia del 95%.

Para verificar, se realiza también utilizando la librería de `scipy` para hacer el cálculo del estadístico y el p-valor.

In [ ]:
# Separar las ventas por tienda, generando los grupos
groups = [group['sales'].values for _, group in df.groupby('store')]
store_names = [name for name, _ in df.groupby('store')]

# ANOVA de un factor (ventas por tienda)
f_stat, p_value = f_oneway(*groups)
alfa = 0.05

print(f"Estadístico F: {f_stat:.4f}")
if p_value < alfa:
    print(f"Resultado: p-valor ({p_value:.4f}) < alfa ({alfa}) -> Se rechaza la hipótesis nula.")
    print("Hay diferencias significativas en las ventas entre las tiendas.")
else:
    print(f"Resultado: p-valor ({p_value:.4f}) >= alfa ({alfa}) -> No se rechaza la hipótesis nula.")
    print("No hay evidencia suficiente para afirmar que las ventas difieren entre las tiendas.")


Estadístico F: 1723.4975
Resultado: p-valor (0.0000) < alfa (0.05) -> Se rechaza la hipótesis nula.
Hay diferencias significativas en las ventas entre las tiendas.


De esta manera, con una significancia del 95%, podemos decir que las ventas esperadas de todas las tiendas no son iguales, o dicho de otra forma, que existe al menos un par de tiendas, tal que las ventas esperadas de la primera difieren de las ventas esperadas de la segunda.

#### 4) 

In [11]:
# Se crea un nuevo dataset de los datos agrupados por tienda, con el cálculo de la media,
# la desviación estándar y el número de observaciones.
df_tiendas = df.groupby('store')['sales'].agg(['mean', 'std', 'count']).reset_index()
df_tiendas

,store,mean,std,count
0,Córdoba,8714.890511,1186.501147,1096
1,La Floresta,7712.862226,1181.888832,1096
2,Los Cedros,8717.111314,1181.648070,1096
3,Palermo,6718.302920,1183.135425,1096
4,Santa Ana,10716.547445,1180.534149,1096


Se recrea en markdown la tabla obtenida.
| Tienda | Media | Desviación estándar | n |
| --- | --- | --- | --- |
| Córdoba | 8714.890511 | 1186.501147 | 1096 |
| La Floresta | 7712.862226 | 1181.888832 | 1096 |
| Los Cedros | 8717.111314 | 1181.648070 | 1096 |
| Palermo | 6718.302920 | 1183.135425 | 1096 |
| Santa Ana | 10716.547445 | 1180.534149 | 1096 |

Se puede ver que la tienda con mayor promedio de ventas es **Santa Ana** con $10716.55$, y la de menor promedio es la tienda de **Palermo**, con $6718.30$. Esto se condice con los efectos de las tiendas indicados en la consigna, repetidos en la siguiente tabla, donde podemos ver que la tienda de Santa Ana tiene el mayor efecto, con $5000$, y la de Palermo el menor, con $1000$.
| Tienda | Efecto |
| --- | --- |
| Santa Ana | 5000 |
| La Floresta | 2000 |
| Los Cedros | 3000 |
| Palermo | 1000 |
| Córdoba | 3000 |

Ahora se debe hacer una prueba de hipótesis para definir si la diferencia entre las medias de estas dos tiendas es distinta de 0 o no. Para ello, planteamos las hipótesis nula y alternativa:
- $H_0: \bar{X_{SA}} - \bar{X_P} = 0$
- $H_1: \bar{X_{SA}} - \bar{X_P} \neq 0$

Considerando las varianzas desconocidas y distintas (ya que al tener distinto $\lambda$ base, probablemente la varianza también), se debe usar la t de Welch, cuyo estadístico será:
$$t = \frac{\bar{X_{SA}} - \bar{X_P}}{\sqrt{\frac{S_{SA}^2}{n_{SA}} + \frac{S_P^2}{n_P}}}$$

Y los grados de libertad se aproximan mediante:
$$\nu = \frac{\left(\frac{S_{SA}^2}{n_{SA}} + \frac{S_P^2}{n_P}\right)^2}{\frac{\left(\frac{S_{SA}^2}{n_{SA}}\right)^2}{n_{SA} - 1} + \frac{\left(\frac{S_P^2}{n_P}\right)^2}{n_P - 1}}$$


No se indica explícitamente en el enunciado, pero se utilizará un niel de significancia de $\alpha=0.05$